# Bài 6 (trang PDF 45, Groceries) – Câu 5

**Sinh viên:** Trần Chí Nguyên  
**MSSV:** 6451071052

Notebook chạy độc lập trên Google Colab. Dữ liệu công khai được tải tự động khi chạy cell đầu tiên.

In [1]:
import warnings
warnings.filterwarnings("ignore")
import csv, requests, time, numpy as np, pandas as pd, seaborn as sns, matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

print("Trần Chí Nguyên - 6451071052")
FILE_ID = "1UIeIAeGPrmXaSsFOtfhw9h5gK4RPkseI"
def load_groceries():
    path = Path("/content/groceries.csv")
    if not path.exists():
        response = requests.get("https://drive.google.com/uc", params={"export": "download", "id": FILE_ID}, timeout=90)
        response.raise_for_status()
        path.write_bytes(response.content)
    with path.open(encoding="utf-8-sig", newline="") as handle:
        return [[item.strip() for item in row if item.strip()] for row in csv.reader(handle)]
transactions = load_groceries()
print(f"Số giao dịch: {len(transactions):,}; số mặt hàng duy nhất: {len(set(x for t in transactions for x in t)):,}")

Trần Chí Nguyên - 6451071052
Số giao dịch: 9,835; số mặt hàng duy nhất: 169


## Thực hiện yêu cầu

Cell dưới đây chứa lời giải cho câu này.

In [2]:
%pip -q install mlxtend
import warnings
warnings.filterwarnings("ignore")
from mlxtend.frequent_patterns import fpgrowth
from mlxtend.preprocessing import TransactionEncoder

encoder = TransactionEncoder()
encoded = encoder.fit(transactions).transform(transactions)
onehot = pd.DataFrame(encoded, columns=encoder.columns_)

MIN_SUPPORTS = np.round(np.arange(0.0001, 0.00101, 0.0001), 4).tolist()
MAX_ITEMSET_LENGTH = 3
fpgrowth_results = []
for min_sup in MIN_SUPPORTS:
    start = time.perf_counter()
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        patterns = fpgrowth(onehot, min_support=min_sup, use_colnames=True, max_len=MAX_ITEMSET_LENGTH)
    elapsed = time.perf_counter() - start
    fpgrowth_results.append({"min_support": min_sup, "patterns": len(patterns), "seconds": elapsed})
    print(f"min_sup={min_sup:.4%}: {len(patterns):,} tập, {elapsed:.2f}s")
fpgrowth_results = pd.DataFrame(fpgrowth_results)
print(f"Giới hạn itemset tối đa: {MAX_ITEMSET_LENGTH} mặt hàng (để so sánh cùng Apriori).")

min_sup=0.0100%: 149,229 tập, 4.49s


min_sup=0.0200%: 70,858 tập, 2.45s


min_sup=0.0300%: 44,260 tập, 1.84s


min_sup=0.0400%: 31,576 tập, 1.58s


min_sup=0.0500%: 24,172 tập, 1.08s


min_sup=0.0600%: 19,251 tập, 1.34s


min_sup=0.0700%: 15,844 tập, 1.06s


min_sup=0.0800%: 13,340 tập, 1.38s


min_sup=0.0900%: 11,438 tập, 0.76s


min_sup=0.1000%: 9,969 tập, 0.75s
Giới hạn itemset tối đa: 3 mặt hàng (để so sánh cùng Apriori).
